### Pip install que precisam ocorrer antes de importe de blibliotecas especificas

In [1]:
# BLOCO 1
%pip install uv
!uv pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install xformers --index-url https://download.pytorch.org/whl/cu121

!uv pip install unsloth unsloth-zoo torchao
!uv pip install accelerate transformers trl peft bitsandbytes datasets
!uv pip install setuptools
!uv pip install pandas scikit-learn google-generativeai matplotlib ipywidgets

/home/cesar/Documents/GitHub/challenge_1_ctrl_alt_del/myenv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
Using Python 3.10.21 environment at: myenv
Checked 3 packages in 8ms
Using Python 3.10.21 environment at: myenv
Checked 1 package in 3ms
Using Python 3.10.21 environment at: myenv
Resolved 99 packages in 604ms                                        
Uninstalled 2 packages in 26ms
Installed 2 packages in 78ms                                
 - fsspec==2026.7.0
 + fsspec==2025.9.0
 - torchao==0.7.0+cu121
 + torchao==0.18.0
Using Python 3.10.21 environment at: myenv
Checked 6 packages in 21ms
Using Python 3.10.21 environment at: myenv
Checked 1 package in 3ms
Using Python 3.10.21 environment at: myenv
Checked 5 packages in 16ms


In [1]:
# BLOCO 2
import os
import time
import torch
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import google.generativeai as genai

# Verificação de Hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Nenhuma GPU'
print(f"Iniciando Pipeline em: {device} | Dispositivo: {gpu_name}")

/home/cesar/Documents/GitHub/challenge_1_ctrl_alt_del/myenv/lib/python3.10/site-packages/unsloth/_gpu_init.py:104: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: Could not load this library: /home/cesar/Documents/GitHub/challenge_1_ctrl_alt_del/myenv/lib/python3.10/site-packages/torchaudio/lib/libtorchaudio.so
  disable_torchaudio_if_cuda_mismatched()
[fla.utils._device|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


In file included from /home/cesar/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/include/python3.10/Python.h:8,
                 from /tmp/tmpdxwo02an/cuda_utils.c:9:
/home/cesar/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/include/python3.10/pyconfig.h:1646:9: warning: ‘_POSIX_C_SOURCE’ redefined
 1646 | #define _POSIX_C_SOURCE 200809L
      |         ^~~~~~~~~~~~~~~
In file included from /usr/include/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from /home/cesar/Documents/GitHub/challenge_1_ctrl_alt_del/myenv/lib/python3.10/site-packages/triton/backends/nvidia/include/cuda.h:56,
                 from /tmp/tmpdxwo02an/cuda_utils.c:1:
/usr/include/features.h:319:10: note: this is the location of the previous definition
  319 | # define _POSIX_C_SOURCE        202405L
      |          ^~~~~~~~~~~~~~~
W0923 17:28:33.896000 32449 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively suppor

🦥 Unsloth Zoo will now patch everything to make training faster!


/home/cesar/Documents/GitHub/challenge_1_ctrl_alt_del/myenv/lib/python3.10/site-packages/unsloth/import_fixes.py:4896: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)
/home/cesar/Documents/GitHub/challenge_1_ctrl_alt_del/myenv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.21) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Iniciando Pipeline em: cuda | Dispositivo: NVIDIA GeForce RTX 3060


/tmp/ipykernel_32449/1322670274.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
import os
import re
import time
import pandas as pd
import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("GOOGLE_API_KEY não encontrada no arquivo .env!")

genai.configure(api_key=api_key)
# Usando o modelo configurado
model = genai.GenerativeModel("gemini-3.5-flash-lite")


def chunk_texto(texto: str, max_chars: int = 1000) -> list[str]:
    """Divide propostas extensas em blocos menores respeitando quebras de linha e pontuação."""
    if not isinstance(texto, str) or not texto.strip():
        return []

    paragrafos = [p.strip() for p in re.split(r"\n+", texto) if p.strip()]
    chunks = []
    chunk_atual = ""

    for p in paragrafos:
        if len(chunk_atual) + len(p) <= max_chars:
            chunk_atual += (" " if chunk_atual else "") + p
        else:
            if chunk_atual:
                chunks.append(chunk_atual)
            if len(p) > max_chars:
                frases = re.split(r"(?<=[.!?]) +", p)
                sub_chunk = ""
                for f in frases:
                    if len(sub_chunk) + len(f) <= max_chars:
                        sub_chunk += (" " if sub_chunk else "") + f
                    else:
                        if sub_chunk:
                            chunks.append(sub_chunk)
                        sub_chunk = f
                if sub_chunk:
                    chunks.append(sub_chunk)
                chunk_atual = ""
            else:
                chunk_atual = p

    if chunk_atual:
        chunks.append(chunk_atual)

    return chunks


def gerar_proposta_falsa(candidato: str, proposta_real: str, max_retries: int = 5) -> str:
    """Gera proposta falsa tratando automaticamente o limite de cota (Erro 429)."""
    prompt = f"""Você é um gerador de dados sintéticos para treinamento de checagem de fatos.
Candidato: {candidato}
Proposta Real: {proposta_real}

Gere UMA proposta FALSA/DISTORCIDA que pareça ter sido dita pelo candidato "{candidato}", baseando-se na proposta real acima.
Aplique um exagero inviável, alteração de público-alvo ou inclusão de custos/regras absurdas.
Retorne APENAS o texto da proposta falsa."""

    for tentativa in range(max_retries):
        try:
            response = model.generate_content(prompt)
            return response.text.strip() if response.text else ""
        except ResourceExhausted:
            tempo_espera = 15 * (tentativa + 1)
            print(f"      [429] Limite da API atingido. Aguardando {tempo_espera}s (tentativa {tentativa + 1}/{max_retries})...")
            time.sleep(tempo_espera)
        except Exception as e:
            print(f"      [ERRO API] Erro ao gerar fake para {candidato}: {e}")
            return ""

    print(f"      [ERRO] Falha após {max_retries} tentativas para {candidato}.")
    return ""


print("A carregar e preparar o ficheiro CSV original...")
df_original = pd.read_csv("./com_propostas.csv", sep=";", encoding="latin1")
df_original.columns = df_original.columns.str.strip().str.upper()

reais_registros = []
falsas_registros = []

total_linhas = len(df_original)
print(f"Processando {total_linhas} candidatos do arquivo...")

for index, row in df_original.iterrows():
    cand_nome = row.get("NM_CANDIDATO", row.get("NM_URNA_CANDIDATO", "Desconhecido"))
    prop_real_bruta = row.get("PROPOSTA", "")

    if pd.isna(prop_real_bruta) or not str(prop_real_bruta).strip():
        continue

    chunks = chunk_texto(str(prop_real_bruta), max_chars=1000)
    print(f"\n[Candidato {index + 1}/{total_linhas}: {cand_nome}] - Dividido em {len(chunks)} parte(s)")

    for idx, chunk in enumerate(chunks):
        # 1. Registro Verdadeiro
        reg_real = row.to_dict()
        reg_real["PROPOSTA"] = chunk
        reg_real["CHUNK_ID"] = idx + 1
        reg_real["LABEL_CATEGORY"] = "true"
        reais_registros.append(reg_real)

        
        print(f"  -> Parte {idx + 1}/{len(chunks)}: Enviando para o Gemini...")
        prop_falsa = gerar_proposta_falsa(str(cand_nome), chunk)

        if prop_falsa:
            print(f"     [SUCESSO] Fake gerada: \"{prop_falsa[:80]}...\"")
            reg_falso = row.to_dict()
            reg_falso["PROPOSTA"] = prop_falsa
            reg_falso["CHUNK_ID"] = idx + 1
            reg_falso["LABEL_CATEGORY"] = "false"
            falsas_registros.append(reg_falso)
        else:
            print("     [AVISO] Nenhuma resposta gerada para esta parte.")

        
        print("     Aguardando 5s antes da próxima chamada...")
        time.sleep(5)

df_reais = pd.DataFrame(reais_registros)
df_falsas = pd.DataFrame(falsas_registros)
df_final = pd.concat([df_reais, df_falsas], ignore_index=True)

print(f"\nConcluído! Trechos reais: {len(df_reais)} | Falsos gerados: {len(df_falsas)}")

arquivo_saida = "./propostas_com_fakes.csv"
df_final.to_csv(arquivo_saida, sep=";", index=False, encoding="utf-8-sig")

print(f"Ficheiro guardado com sucesso em: {arquivo_saida}")

A carregar e preparar o ficheiro CSV original...
Processando 412 candidatos do arquivo...

[Candidato 1/412: SEBASTIAO BOCALOM RODRIGUES] - Dividido em 52 parte(s)
  -> Parte 1/52: Enviando para o Gemini...
     [SUCESSO] Fake gerada: "Vamos zerar completamente o isolamento geográfico do Acre até o final do primeir..."
     Aguardando 5s antes da próxima chamada...
  -> Parte 2/52: Enviando para o Gemini...
     [SUCESSO] Fake gerada: "Assumo o compromisso de construir, logo no primeiro ano de mandato, uma ferrovia..."
     Aguardando 5s antes da próxima chamada...
  -> Parte 3/52: Enviando para o Gemini...
     [SUCESSO] Fake gerada: "Vamos proibir qualquer atividade de preservação ambiental no Acre e transformar ..."
     Aguardando 5s antes da próxima chamada...
  -> Parte 4/52: Enviando para o Gemini...
     [SUCESSO] Fake gerada: "Vamos transformar a economia do Acre na maior potência industrial do Pacífico em..."
     Aguardando 5s antes da próxima chamada...
  -> Parte 5/52: Env

In [ ]:
# BLOCO 4
max_seq_length = 4096
lora_rank = 32         

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/home/cesar/Documents/GitHub/challenge_1_ctrl_alt_del/modelo_propostas_gguf_v3_gguf/mistral-7b-instruct-v0.3.Q4_K_M.gguf",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_rank,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# NAO RODE ISSO, IGNORE
FONTE_TSE = "https://dadosabertos.tse.jus.br/dataset/candidatos-2026"
df_final["FONTE"] = FONTE_TSE

print("A guardar os ficheiros CSV em latin1...")

# 1. Salvar apenas as falsas
df_falsas_export = df_final[df_final["LABEL_CATEGORY"] == "false"]
df_falsas_export.to_csv(
    "dataset_apenas_falsas.csv",
    index=False,
    sep=";",
    encoding="latin1",
    errors="replace",
)

# 2. Salvar completo e agrupado por candidato
df_final_agrupado = df_final.sort_values(
    by=["NM_CANDIDATO", "LABEL_CATEGORY"], ascending=[True, False]
)
df_final_agrupado.to_csv(
    "dataset_completo_agrupado.csv",
    index=False,
    sep=";",
    encoding="latin1",
    errors="replace",
)

# 3. Salvar embaralhado para treino do modelo ML
df_final_treino = df_final.sample(frac=1, random_state=42).reset_index(
    drop=True
)
df_final_treino.to_csv(
    "dataset_treino_embaralhado.csv",
    index=False,
    sep=";",
    encoding="latin1",
    errors="replace",
)

print("Concluído! Ficheiros guardados na pasta atual com encoding latin1.")

In [ ]:
# BLOCO 5
prompt_template = """### Instrução:
Classifique o texto a seguir atribuído ao candidato {candidato} (Fonte: {fonte}) como 'verdadeiro' (true) ou 'falso' (false).

### Candidato:
{candidato}

### Fonte:
{fonte}

### Conteúdo / Proposta:
{proposta}

### Resposta:
{resposta}"""


def formatar_prompts(dataframe):
    textos = []
    for _, row in dataframe.iterrows():
        candidato = row.get("NM_CANDIDATO")
        if pd.isna(candidato) or not str(candidato).strip():
            candidato = row.get("NM_URNA_CANDIDATO", "Desconhecido")

        fonte = row.get("FONTE", "Não informada")
        proposta = row.get("PROPOSTA", "")

        texto = (
            prompt_template.format(
                candidato=candidato,
                fonte=fonte,
                proposta=proposta,
                resposta=row["LABEL_CATEGORY"],
            )
            + tokenizer.eos_token
        )
        textos.append(texto)
    return pd.DataFrame({"text": textos})


# Divisão de Treino e Validação (80/20)
train_df, val_df = train_test_split(
    df_final_treino, test_size=0.2, random_state=42, stratify=df_final_treino["LABEL_CATEGORY"]
)

dataset_treino = Dataset.from_pandas(formatar_prompts(train_df))
dataset_validacao = Dataset.from_pandas(formatar_prompts(val_df))

In [ ]:
# BLOCO 6
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_treino,
    eval_dataset = dataset_validacao,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, 
    args = SFTConfig(
        per_device_train_batch_size = 4,   
        gradient_accumulation_steps = 4,     
        warmup_ratio = 0.05,
        num_train_epochs = 4,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_propostas",
    ),
)

# Iniciar Treinamento SFT
trainer_stats = trainer.train()
print("Treinamento finalizado com sucesso!")

In [ ]:
# BLOCO 7
gguf_directory = "modelo_propostas_gguf_v3"
quantization_method = "q4_k_m"
print(f"Exportando modelo para GGUF em quantização {quantization_method}...")

model.save_pretrained_gguf(
    gguf_directory, 
    tokenizer, 
    quantization_method = quantization_method
)

print(f"Modelo GGUF salvo com sucesso na pasta: ./{gguf_directory}")